# {title}

依次运行全部单元。目标是验证固定版本的 `inference_instruct2` 能在当前 Colab GPU 上生成一条 WAV，并保存可追溯记录。**当前验证状态：{status}。未经 Colab GPU 实测，不能视为验证通过。**

先在“运行时 → 更改运行时类型”选择 GPU。模型约 9.8 GB；首次下载和依赖安装会耗时。默认把模型放在 `/content/models/FunAudioLLM/Fun-CosyVoice3-0.5B-2512`，这是 Colab 临时目录，重启后需重新下载。若要持久保存，先挂载 Google Drive，再将下面的 `MODEL_ROOT` 改为 Drive 中专用模型目录。不要把模型下载到代码目录。

本笔记本固定上游 CosyVoice 源码 commit `074ca6d`、ModelScope 模型 revision `3a9eb0e`。复用 Colab 自带的 PyTorch/CUDA，使用官方非 vLLM 推理路径；实际版本写入日志。运行结果取决于当次 Colab GPU、驱动和空间；安装或推理失败时停下并保留日志。
官方源码：[CosyVoice]({source_url})；模型：[ModelScope]({model_url})；许可：{license}。


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
MODEL_ROOT = Path('/content/models')
WORK = Path('/content/cosyvoice3_colab')
MODEL = MODEL_ROOT / 'FunAudioLLM' / 'Fun-CosyVoice3-0.5B-2512'
WORK.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['MODELSCOPE_CACHE'] = str(MODEL_ROOT / 'modelscope_cache')
os.environ['HF_HOME'] = str(MODEL_ROOT / 'huggingface_cache')
assert Path('/content').exists(), '此笔记本面向托管 Colab 运行时'
assert shutil.which('nvidia-smi'), '请先在运行时设置中选择 GPU'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'], text=True))
free_gb=shutil.disk_usage(MODEL_ROOT).free/2**30
print(f'Model root: {MODEL_ROOT}; free space: {free_gb:.1f} GiB')
assert free_gb >= 18 or (MODEL/'cosyvoice3.yaml').exists(), '首次下载与安装建议至少 18 GiB 可用空间'


## 安装缺少的推理依赖
复用 Colab 自带的 PyTorch、torchaudio 和 CUDA。安装时固定这些现有版本；如果依赖不兼容，停止并保留日志，不自动替换 PyTorch。当前采用 `load_vllm=False`。


In [ ]:
import subprocess, sys
import torch, torchaudio
PY = sys.executable
assert torch.cuda.is_available(), 'Colab GPU 未就绪'
TORCH_VERSION, AUDIO_VERSION = torch.__version__, torchaudio.__version__
print('Using Colab Python:', sys.version)
print('Using Colab torch:', TORCH_VERSION, 'CUDA:', torch.version.cuda)
print('Using Colab torchaudio:', AUDIO_VERSION)

def run(args, log_name):
    with (WORK / log_name).open('w') as log:
        result = subprocess.run(args, stdout=log, stderr=subprocess.STDOUT, text=True)
    print(log_name, 'exit=', result.returncode)
    if result.returncode:
        print((WORK / log_name).read_text()[-5000:])
        raise RuntimeError(f'步骤失败，查看 {WORK / log_name}')

constraints = WORK / 'colab_torch_constraints.txt'
constraints.write_text('torch==' + TORCH_VERSION + '\ntorchaudio==' + AUDIO_VERSION + '\n')
requirements = WORK / 'runtime_requirements.txt'
packages = [
    'transformers==4.51.3', 'conformer==0.3.2', 'diffusers==0.29.0',
    'hydra-core==1.3.2', 'HyperPyYAML==1.2.3', 'inflect==7.3.1',
    'librosa>=0.10.2,<1', 'modelscope>=1.20,<2', 'onnxruntime-gpu>=1.22,<2',
    'omegaconf>=2.3,<3', 'openai-whisper==20250625', 'soundfile>=0.12,<1',
    'wetext==0.0.4', 'x-transformers==2.11.24',
    'gdown>=5,<6', 'matplotlib>=3.7,<4', 'wget==3.2',
]
requirements.write_text('\n'.join(packages) + '\n')
run([PY, '-m', 'pip', 'install', '-c', str(constraints), '-r', str(requirements)], 'install_runtime.log')
import importlib.metadata as metadata
assert metadata.version('torch') == TORCH_VERSION
assert metadata.version('torchaudio') == AUDIO_VERSION


## 获取固定源码与模型

ModelScope 下载目标明确设在 `MODEL`，已有完整文件会复用。首轮模型完整性以固定文件 SHA-256 核对；失败时停止，不覆盖现有文件。

In [ ]:
REPO=WORK/'CosyVoice'
if not REPO.exists():
    run(['git','clone','--filter=blob:none','https://github.com/FunAudioLLM/CosyVoice.git',str(REPO)],'clone.log')
run(['git','-C',str(REPO),'checkout','074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc'],'checkout.log')
run(['git','-C',str(REPO),'submodule','update','--init','--recursive','third_party/Matcha-TTS'],'submodule.log')
print('CosyVoice commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())


In [ ]:
download_script = WORK/'download_model.py'
download_script.write_text(r'''from modelscope import snapshot_download
from pathlib import Path
model=Path(__import__('sys').argv[1])
model.parent.mkdir(parents=True, exist_ok=True)
snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', revision='3a9eb0e104c9ce5bb7ef0af4ac86d9b79abc4029', local_dir=str(model))
''')
if not (MODEL/'cosyvoice3.yaml').exists():
    run([PY,str(download_script),str(MODEL)],'download_model.log')
assert (MODEL/'cosyvoice3.yaml').exists()


In [ ]:
import hashlib
expected_sha256 = {'llm.pt': '69f43bd545131c30e98947fb360ea8b4dc9916d8e83dded7757c7ea4f5a24970', 'llm.rl.pt': '74d34b01a80c7154670ae75ac372d1b1712c78bceae9f467eb9f1f6f61ec764f', 'flow.pt': 'a6fab32a7825e5b0bc855ddd948f8db9370b0a786fbc249caa4595e95b608e4b', 'hift.pt': 'b279d7641eb97ae55b3b540cfba4f953c26492a2df758328a89a4d007ab87a65', 'speech_tokenizer_v3.onnx': '23236a74175dbdda47afc66dbadd5bcb41303c467a57c261cb8539ad9db9208d', 'speech_tokenizer_v3.batch.onnx': 'b156b8a7bbff436585e153f4637b9a368009005ac66efa108a6c8bfb34e5ee43', 'flow.decoder.estimator.fp32.onnx': '9b51b9533a55937762b262bf2cf9c6220ce40760f76d6532cb16a6a6d84059a8', 'CosyVoice-BlankEN/model.safetensors': '130282af0dfa9fe5840737cc49c9a0d339d06075f83c5a315c3372c9a0740d0b96'}
for relative, expected in expected_sha256.items():
    path=MODEL/relative
    assert path.is_file(), f'缺少模型文件: {path}'
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):
            h.update(block)
    assert h.hexdigest()==expected, f'SHA-256 不一致: {path}'
    print(relative, 'SHA-256 OK')


## 单条合成

使用用户输入文本，固定参考音频、指令、语速 1.0 与随机种子。采用官方 `AutoModel(..., load_vllm=False)` 和 `inference_instruct2`，不修改上游核心代码。首次加载与输入处理另计，不将单次耗时视为稳定性能。


In [ ]:
check_script=WORK/'check_env.py'
check_script.write_text(r'''import sys, torch, torchaudio, modelscope, transformers, onnxruntime, whisper
sys.path.insert(0, sys.argv[1]+'/third_party/Matcha-TTS')
sys.path.insert(0, sys.argv[1])
from cosyvoice.cli.cosyvoice import AutoModel
from whisper.tokenizer import get_tokenizer
print('Python',sys.version)
print('PyTorch',torch.__version__,'CUDA',torch.version.cuda,'available',torch.cuda.is_available())
print('torchaudio',torchaudio.__version__,'transformers',transformers.__version__,'modelscope',modelscope.__version__,'onnxruntime',onnxruntime.__version__)
assert torch.cuda.is_available()
''')
run([PY,str(check_script),str(REPO)],'environment.log')
run([PY,'-m','pip','freeze'],'pip_freeze.log')
print((WORK/'environment.log').read_text()[-2500:])


## 输入文本
修改下一单元的文本后执行。参考音频沿用官方示例，当前只演示 instruct2 单句合成。


In [ ]:
TEXT = '你好，欢迎体验 CosyVoice3。'
assert TEXT.strip(), '请输入非空文本'
print('输入字符数:', len(TEXT))


In [ ]:
synth_script=WORK/'synthesize.py'
synth_script.write_text(r'''import hashlib, json, sys, time
from pathlib import Path
import numpy as np, soundfile as sf, torch
repo, model, work = map(Path, sys.argv[1:4])
sys.path.insert(0,str(repo/'third_party/Matcha-TTS'))
sys.path.insert(0,str(repo))
from cosyvoice.cli.cosyvoice import AutoModel
torch.manual_seed(1986)
model_obj=AutoModel(model_dir=str(model),load_vllm=False,load_trt=False,fp16=False)
text=sys.argv[4]
instruction='You are a helpful assistant. 请用自然、清晰的语气朗读。<|endofprompt|>'
prompt=repo/'asset/zero_shot_prompt.wav'
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start=time.monotonic()
chunks=[x['tts_speech'].detach().cpu() for x in model_obj.inference_instruct2(text,instruction,str(prompt),stream=False,speed=1.0,text_frontend=True)]
torch.cuda.synchronize()
elapsed=time.monotonic()-start
assert chunks, '模型没有返回音频'
audio=torch.cat(chunks,dim=-1).squeeze().numpy().astype(np.float32,copy=False)
out=work/'cosyvoice3_smoke.wav'
sf.write(out,audio,int(model_obj.sample_rate),format='WAV',subtype='PCM_16')
record={'text':text,'instruction':instruction,'prompt_wav':str(prompt),'speed':1.0,'seed':1986,'sample_rate':int(model_obj.sample_rate),'samples':len(audio),'seconds':len(audio)/model_obj.sample_rate,'inference_elapsed_seconds':elapsed,'rtf_single_call':round(elapsed/(len(audio)/model_obj.sample_rate),3),'gpu_peak_allocated_mib':round(torch.cuda.max_memory_allocated()/2**20,1),'gpu_peak_reserved_mib':round(torch.cuda.max_memory_reserved()/2**20,1),'audio_path':str(out),'audio_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),'model_revision':'3a9eb0e104c9ce5bb7ef0af4ac86d9b79abc4029','code_revision':'074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc'}
(work/'result.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(json.dumps(record,ensure_ascii=False,indent=2))
''')
run([PY,str(synth_script),str(REPO),str(MODEL),str(WORK),TEXT],'synthesis.log')
print((WORK/'result.json').read_text())


## 试听与保存

成功后试听 WAV，并下载包含环境、下载和推理日志的压缩包。Colab 运行时可能被回收；下载结果后再结束会话。

In [ ]:
from IPython.display import Audio, display
import json, zipfile
record=json.loads((WORK/'result.json').read_text())
display(Audio(filename=record['audio_path']))
archive=WORK/'cosyvoice3_colab_result.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in [WORK/'cosyvoice3_smoke.wav',WORK/'result.json',*WORK.glob('*.log')]:
        z.write(p,p.name)
from google.colab import files
files.download(str(archive))
